# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path('.env'))


True

In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob
from pathlib import Path

price_data_dir = Path(os.environ['PRICE_DATA']).expanduser().resolve()
price_files = sorted(glob(str(price_data_dir / '**' / '*.parquet'), recursive=True))
len(price_files)


3040

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [4]:
import pandas as pd

dd_price = dd.read_parquet(price_files)
dd_price = dd_price.rename(columns={'Adj Close': 'Adj_Close', 'ticker': 'Ticker'})

meta = dd_price._meta.assign(
    Close_lag_1=pd.Series(dtype='float64'),
    Adj_Close_lag_1=pd.Series(dtype='float64'),
    returns=pd.Series(dtype='float64'),
    hi_lo_range=pd.Series(dtype='float64')
)

def add_features(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values(['Ticker', 'Date']).copy()
    pdf['Close_lag_1'] = pdf.groupby('Ticker')['Close'].shift(1)
    pdf['Adj_Close_lag_1'] = pdf.groupby('Ticker')['Adj_Close'].shift(1)
    pdf['returns'] = pdf['Close'] / pdf['Close_lag_1'] - 1
    pdf['hi_lo_range'] = pdf['High'] - pdf['Low']
    return pdf

dd_feat = dd_price.map_partitions(add_features, meta=meta)
dd_feat


,Date,Open,High,Low,Close,Adj_Close,Volume,source,Ticker,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
npartitions=3040,,,,,,,,,,,,,,
,datetime64[ns],float64,float64,float64,float64,float64,float64,string,string,int32,float64,float64,float64,float64
,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [5]:
df_feat = dd_feat.compute()
df_feat = df_feat.drop_duplicates(subset=['Ticker', 'Date']).sort_values(['Ticker', 'Date']).reset_index(drop=True)
df_feat['ma_returns_10'] = (
    df_feat.groupby('Ticker', group_keys=False)['returns']
          .transform(lambda s: s.rolling(10, min_periods=1).mean())
)
df_feat.head()


,Date,Open,High,Low,Close,Adj_Close,Volume,source,Ticker,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,ma_returns_10
0,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,ACN,2001,NaN,NaN,NaN,0.29,NaN
1,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,ACN,2001,15.17,11.404394,-0.010547,0.25,-0.010547
2,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,ACN,2001,15.01,11.284108,-0.000666,0.46,-0.005607
3,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,ACN,2001,15.00,11.276587,-0.009333,0.27,-0.006849
4,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,ACN,2001,14.86,11.171341,0.006057,0.30,-0.003623


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

In [6]:
#It was not necesary to convert to pandas to calculate moving average return
#that dataset is not that big that we need to use dask to work with the data set

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

ðŸš¨ **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** ðŸš¨ for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.